# 15. 당뇨병 예측 모델 개선과 실험 관리

이번 장에서는 모델을 하나만 만드는 것이 아니라, 여러 실험을 비교합니다.

핵심 목표:

```text
baseline 모델
dropout을 더 강하게 넣은 모델
조금 더 넓은 모델
```

이 세 모델을 같은 데이터, 같은 평가 기준으로 비교합니다.

## 1. 라이브러리 불러오기

이번 장에서는 `EarlyStopping`을 새로 사용합니다.

`EarlyStopping`은 검증 성능이 더 이상 좋아지지 않을 때 학습을 일찍 멈추는 콜백입니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout

# EarlyStopping은 학습 중 검증 손실이 더 이상 좋아지지 않으면 학습을 멈추는 도구입니다.
from keras.callbacks import EarlyStopping

## 2. 데이터 준비

13장, 14장과 같은 당뇨병 이진 분류 데이터를 사용합니다.

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\diabetes_or_cardiovascular\diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
TARGET = "Diabetes_binary"

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

input_dim = X_train_scaled.shape[1]

print("input_dim:", input_dim)
print("학습 데이터:", X_train_scaled.shape)
print("검증 데이터:", X_val_scaled.shape)

## 3. 모델 생성 함수 만들기

실험을 여러 번 할 때는 모델 만드는 코드를 함수로 묶으면 좋습니다.

함수는 “레시피”와 비슷합니다.

원하는 설정을 넣으면 그 설정에 맞는 새 모델을 만들어 줍니다.

In [ ]:
def build_mlp_model(input_dim, hidden_units=(32, 16), dropout_rate=0.2):
    """설정값에 따라 MLP 모델을 새로 만듭니다."""
    
    model = Sequential()
    
    # Input은 모델이 받을 입력 특성 개수를 알려줍니다.
    model.add(Input(shape=(input_dim,)))
    
    # hidden_units에 들어 있는 숫자만큼 Dense 층을 반복해서 추가합니다.
    for units in hidden_units:
        model.add(Dense(units, activation="relu"))
        
        # dropout_rate가 0보다 클 때만 Dropout 층을 추가합니다.
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    
    # 이진 분류 출력층입니다.
    model.add(Dense(1, activation="sigmoid"))
    
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    
    return model

## 4. 평가 함수 만들기

모든 실험을 같은 기준으로 평가해야 공정하게 비교할 수 있습니다.

이번 장에서는 threshold 0.5를 기준으로 accuracy, precision, recall, f1을 계산합니다.

In [ ]:
def evaluate_model(model, X_val_scaled, y_val, threshold=0.5):
    """모델의 예측 확률을 0/1로 바꾸고 주요 지표를 계산합니다."""
    
    # predict()는 각 샘플에 대한 예측 확률을 반환합니다.
    y_prob = model.predict(X_val_scaled, verbose=0).ravel()
    
    # threshold 이상이면 1, 아니면 0으로 바꿉니다.
    y_pred = (y_prob >= threshold).astype(int)
    
    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_val, y_pred),
    }

## 5. 실험 설정 목록 만들기

실험 설정을 표처럼 리스트에 적어 둡니다.

이렇게 하면 어떤 모델을 비교했는지 나중에 보기 쉽습니다.

In [ ]:
experiments = [
    {
        "name": "baseline",
        "hidden_units": (32, 16),
        "dropout_rate": 0.2,
    },
    {
        "name": "dropout_0_4",
        "hidden_units": (32, 16),
        "dropout_rate": 0.4,
    },
    {
        "name": "wider_model",
        "hidden_units": (64, 32),
        "dropout_rate": 0.2,
    },
]

experiments

## 6. EarlyStopping 설정

검증 손실이 좋아지지 않으면 학습을 멈추도록 설정합니다.

`restore_best_weights=True`는 가장 좋았던 시점의 가중치로 되돌려 줍니다.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",          # 검증 손실을 지켜봅니다.
    patience=3,                  # 3번 연속 좋아지지 않으면 멈춥니다.
    restore_best_weights=True    # 가장 좋았던 가중치로 되돌립니다.
)

## 7. 실험 반복 실행

아래 셀은 실험 목록을 하나씩 꺼내 모델을 만들고 학습한 뒤, 평가 결과를 저장합니다.

나중에 결과표로 비교할 수 있도록 리스트에 기록합니다.

In [ ]:
results = []
histories = {}
models = {}

for experiment in experiments:
    name = experiment["name"]
    
    print("=" * 60)
    print("실험 시작:", name)
    
    # 실험 설정에 맞는 새 모델을 만듭니다.
    model = build_mlp_model(
        input_dim=input_dim,
        hidden_units=experiment["hidden_units"],
        dropout_rate=experiment["dropout_rate"],
    )
    
    # fit()은 모델을 학습합니다.
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=30,
        batch_size=64,
        callbacks=[early_stopping],
        verbose=1
    )
    
    # 검증 데이터에서 평가 지표를 계산합니다.
    metrics = evaluate_model(model, X_val_scaled, y_val, threshold=0.5)
    
    # 결과표에 넣을 값들을 저장합니다.
    results.append({
        "experiment": name,
        "hidden_units": str(experiment["hidden_units"]),
        "dropout_rate": experiment["dropout_rate"],
        "epochs_ran": len(history.history["loss"]),
        "best_val_loss": min(history.history["val_loss"]),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
    })
    
    histories[name] = history
    models[name] = model

print("모든 실험 완료")

## 8. 결과표 보기

이제 실험 결과를 표로 비교합니다.

이 표가 프로젝트 보고서의 핵심 근거가 됩니다.

In [ ]:
results_df = pd.DataFrame(results)

# f1 기준으로 높은 모델이 위에 오도록 정렬합니다.
results_df = results_df.sort_values("f1", ascending=False)

results_df

In [ ]:
plt.figure(figsize=(9, 4))

x = np.arange(len(results_df))
width = 0.2

plt.bar(x - width, results_df["precision"], width=width, label="precision")
plt.bar(x, results_df["recall"], width=width, label="recall")
plt.bar(x + width, results_df["f1"], width=width, label="f1")

plt.xticks(x, results_df["experiment"], rotation=20)
plt.ylim(0, 1)
plt.title("Experiment Comparison")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()

## 9. 학습 곡선 비교

모델별로 검증 손실이 어떻게 변했는지 봅니다.

검증 손실이 낮고 안정적인 모델이 좋은 후보가 될 수 있습니다.

In [ ]:
plt.figure(figsize=(8, 5))

for name, history in histories.items():
    # history.history["val_loss"]에는 epoch별 검증 손실이 저장되어 있습니다.
    plt.plot(history.history["val_loss"], label=name)

plt.title("Validation Loss by Experiment")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 10. 가장 좋은 모델 선택하기

여기서는 f1-score가 가장 높은 모델을 선택합니다.

하지만 실제 프로젝트에서는 목적에 따라 recall이 더 중요한 기준이 될 수도 있습니다.

In [ ]:
# 정렬된 results_df의 첫 번째 행이 f1 기준 가장 좋은 실험입니다.
best_experiment_name = results_df.iloc[0]["experiment"]
best_model = models[best_experiment_name]

print("선택된 모델:", best_experiment_name)

best_metrics = evaluate_model(best_model, X_val_scaled, y_val, threshold=0.5)

print("Confusion Matrix:")
display(pd.DataFrame(
    best_metrics["confusion_matrix"],
    index=["actual 0", "actual 1"],
    columns=["pred 0", "pred 1"]
))

## 11. 결과표와 모델 저장

실험 결과표와 최종 모델을 저장합니다.

프로젝트에서는 모델 파일뿐 아니라 실험 결과표도 같이 남겨야 합니다.

In [ ]:
OUTPUT_DIR = Path(r"C:\work\deepLearning\model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_DIR / "diabetes_experiment_results.csv"
model_path = OUTPUT_DIR / "diabetes_best_experiment_model.keras"

# to_csv()는 DataFrame을 CSV 파일로 저장합니다.
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

# save()는 Keras 모델을 파일로 저장합니다.
best_model.save(model_path)

print("결과표 저장:", results_path)
print("모델 저장:", model_path)

## 12. 해석 문장 만들기

실험 결과는 반드시 말로 해석해야 합니다.

아래 템플릿을 보고 실제 결과에 맞게 고치면 됩니다.

In [ ]:
print("해석 템플릿")
print("- baseline, dropout_0_4, wider_model을 같은 데이터 분할과 같은 평가 기준으로 비교했다.")
print("- EarlyStopping을 사용해 검증 손실이 개선되지 않을 때 학습을 멈췄다.")
print("- 최종 모델은 f1-score를 기준으로 선택했다.")
print("- 건강 위험 예측에서는 필요에 따라 f1 대신 recall을 우선 기준으로 둘 수도 있다.")
print("- 실험 결과표를 저장해 이후 개선 실험과 비교할 수 있게 했다.")

## 13. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. baseline이 있어야 개선 여부를 판단할 수 있다.
2. 실험은 한 번에 너무 많은 것을 바꾸지 않는 것이 좋다.
3. EarlyStopping은 검증 성능이 멈췄을 때 학습을 조기 종료한다.
4. 실험 결과는 표로 남겨야 비교할 수 있다.
5. 최종 모델 선택 기준은 프로젝트 목적에 따라 달라질 수 있다.
```

## 과제

1. 세 실험 중 어떤 모델을 최종 선택할지 고르고 이유를 적어보세요.
2. f1이 아니라 recall을 기준으로 고르면 선택이 달라질 수 있는지 확인해보세요.
3. 다음 실험으로 바꿔보고 싶은 설정을 하나만 고르고 이유를 적어보세요.
4. 실험 결과를 포트폴리오에 설명하는 문장 3개를 작성해보세요.